# Range sensors: the beam model, notebook

The robot of the Note stands at pose (2, 2, 0) in a 5 m x 4 m room. Its laser reports ranges up to 5 m. We build the four parts of the beam model, check the Note's numbers, see why short readings fall off exponentially, and fit the model to 1000 readings with EM.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from math import erf, sqrt

ZMAX, STEP = 5.0, 0.01            # longest reading (m); the laser reports 1 cm steps
SIG_HIT, LAM = 0.05, 2.0          # measurement-noise spread (m), short-reading rate (1/m)
W = dict(hit=0.75, short=0.12, max=0.05, rand=0.08)   # mixture weights
ROOM = (5.0, 4.0)

def gauss(z, mu, sig):
    return np.exp(-0.5 * ((z - mu) / sig) ** 2) / (sig * np.sqrt(2 * np.pi))

def p_hit(z, zs, sig=SIG_HIT):
    area = 0.5 * (erf((ZMAX - zs) / (sig * sqrt(2))) - erf(-zs / (sig * sqrt(2))))   # Gaussian area inside [0, z_max]
    z = np.asarray(z, float)
    return np.where((z >= 0) & (z <= ZMAX), gauss(z, zs, sig) / area, 0.0)

def p_short(z, zs, lam=LAM):
    z = np.asarray(z, float)
    eta = 1 / (1 - np.exp(-lam * zs))
    return np.where((z >= 0) & (z <= zs), eta * lam * np.exp(-lam * z), 0.0)

def p_max(z):                      # the spike at z_max, one 1 cm step wide: height 100, area 1
    z = np.asarray(z, float)
    return np.where(z >= ZMAX - STEP / 2, 1 / STEP, 0.0)

def p_rand(z):
    z = np.asarray(z, float)
    return np.where((z >= 0) & (z <= ZMAX), 1 / ZMAX, 0.0)

def mixture(z, zs, w=W, sig=SIG_HIT, lam=LAM):
    return w['hit'] * p_hit(z, zs, sig) + w['short'] * p_short(z, zs, lam) + w['max'] * p_max(z) + w['rand'] * p_rand(z)

## 1. Ray casting: the expected range of each beam (Note, Section 2.2)

In [2]:
def raycast(pose, ang, step=0.001):
    x, y, th = pose
    d = 0.0
    while d < ZMAX:
        px, py = x + d * np.cos(th + ang), y + d * np.sin(th + ang)
        if not (0 < px < ROOM[0] and 0 < py < ROOM[1]):   # left the room: the beam is inside a wall
            return round(d, 2)
        d += step
    return ZMAX

angles = np.radians([0, 90, 180, 270])
for name, pose in [('A', (2, 2, 0)), ('B', (3, 2, 0))]:
    print(name, [raycast(pose, a) for a in angles])

A [3.0, 2.0, 2.0, 2.0]
B [2.0, 2.0, 3.0, 2.0]


## 2. A Gaussian alone, and the beam model, on the scan of Figure 1 (Note, Sections 3 and 5)

In [3]:
readings = [1.00, 1.97, 2.04, 1.99]
rows = []
for name, pose in [('A (true)', (2, 2, 0)), ('B', (3, 2, 0))]:
    zs = [raycast(pose, a) for a in angles]
    g = [gauss(r, e, SIG_HIT) for r, e in zip(readings, zs)]
    m = [float(mixture(r, e)) for r, e in zip(readings, zs)]
    rows.append(dict(pose=name, expected=zs, gaussian=np.round(g, 4).tolist(), gaussian_product=np.prod(g),
                     mixture=np.round(m, 4).tolist(), mixture_product=np.prod(m)))
pd.DataFrame(rows)

,pose,expected,gaussian,gaussian_product,mixture,mixture_product
0,A (true),"[3.0, 2.0, 2.0, 2.0]","[0.0, 6.6645, 5.7938, 7.8209]",0.000000e+00,"[0.0486, 5.0191, 4.3614, 5.8862]",6.257142
1,B,"[2.0, 2.0, 3.0, 2.0]","[0.0, 6.6645, 0.0, 7.8209]",4.100587e-164,"[0.0491, 5.0191, 0.0201, 5.8862]",0.029102


In [4]:
print('Gaussian density of 1.00 m when 3 m is expected:', gauss(1.0, 3.0, 0.05))   # e^-800 underflows to 0
print('its exponent:', -0.5 * ((1.0 - 3.0) / 0.05) ** 2)

Gaussian density of 1.00 m when 3 m is expected: 0.0
its exponent: -800.0


## 3. Why short readings fall off exponentially (Note, Section 4.2)

Each 0.1 m cell of the 3 m beam is blocked with probability 0.2. We record where the beam first stops.

In [5]:
rng = np.random.default_rng(3)
P, N = 0.2, 30
trials = rng.random((5000, N)) < P
first = np.where(trials.any(1), trials.argmax(1), N)
share = np.bincount(first, minlength=N + 1)[:N] / len(first)
k = np.arange(1, N + 1)
pd.DataFrame(dict(cell=k[:6], simulated=share[:6].round(3), formula=(P * (1 - P) ** (k[:6] - 1)).round(3)))

,cell,simulated,formula
0,1,0.206,0.200
1,2,0.157,0.160
2,3,0.131,0.128
3,4,0.100,0.102
4,5,0.084,0.082
5,6,0.063,0.066


In [6]:
# Bit strings: count the 4-cell strings by their first blocked cell
from itertools import product
counts = {}
for s in product([0, 1], repeat=4):
    f = s.index(1) + 1 if 1 in s else 'none'
    counts[f] = counts.get(f, 0) + 1
counts

{'none': 1, 4: 1, 3: 2, 2: 4, 1: 8}

## 4. Fitting the model to 1000 readings with EM (Note, Section 6)

The readings were simulated from the model with the weights above, the wall 3 m away; `data/wall_readings.csv` holds them.

In [7]:
z = np.loadtxt('data/wall_readings.csv', delimiter=',', skiprows=1)
print(len(z), 'readings;', (z == 5.0).sum(), 'at max range')

def em(z, zs=3.0, iters=60):
    w, sig, lam = dict(hit=.25, short=.25, max=.25, rand=.25), 0.3, 0.5
    log = []
    for it in range(iters + 1):
        parts = np.vstack([w['hit'] * p_hit(z, zs, sig), w['short'] * p_short(z, zs, lam),
                           w['max'] * p_max(z), w['rand'] * p_rand(z)])
        log.append(dict(round=it, **{k: round(v, 3) for k, v in w.items()}, sigma=round(sig, 4), lam=round(lam, 3),
                        loglik=round(np.log(parts.sum(0)).sum(), 1)))
        if it == iters:
            break
        e = parts / parts.sum(0)                                   # E-step: responsibilities
        w = dict(hit=e[0].mean(), short=e[1].mean(), max=e[2].mean(), rand=e[3].mean())   # M-step
        sig = np.sqrt((e[0] * (z - zs) ** 2).sum() / e[0].sum())
        lam = e[1].sum() / (e[1] * z).sum()
    return pd.DataFrame(log)

log = em(z)
log.iloc[[0, 1, 2, 3, 5, 10, 60]]

1000 readings; 47 at max range


,round,hit,short,max,rand,sigma,lam,loglik
0,0,0.250,0.250,0.250,0.250,0.3000,0.500,-932.2
1,1,0.621,0.163,0.047,0.169,0.0805,0.882,280.0
2,2,0.727,0.135,0.047,0.092,0.0519,1.525,444.9
3,3,0.732,0.143,0.047,0.078,0.0506,1.749,447.9
5,5,0.733,0.148,0.047,0.073,0.0506,1.798,448.0
10,10,0.733,0.149,0.047,0.071,0.0506,1.768,448.1
60,60,0.733,0.150,0.047,0.070,0.0506,1.754,448.2


The E-step of round 0 for three readings, with the starting guess (all weights 0.25, sigma 0.3 m, lambda 0.5):

In [8]:
w0 = dict(hit=.25, short=.25, max=.25, rand=.25)
for r in [3.02, 1.00, 5.00]:
    parts = np.array([w0['hit'] * p_hit(r, 3, .3), w0['short'] * p_short(r, 3, .5), w0['max'] * p_max(r), w0['rand'] * p_rand(r)], float)
    print(r, 'weighted parts', parts.round(4), 'responsibilities', (parts / parts.sum()).round(3))

3.02 weighted parts [0.3317 0.     0.     0.05  ] responsibilities [0.869 0.    0.    0.131]
1.0 weighted parts [0.     0.0976 0.     0.05  ] responsibilities [0.    0.661 0.    0.339]
5.0 weighted parts [ 0.    0.   25.    0.05] responsibilities [0.    0.    0.998 0.002]


Because the data is simulated, we know which cause made each reading, so we can check EM against the truth of this sample.

In [9]:
rng = np.random.default_rng(0)
cause = rng.choice(4, size=1000, p=[W['hit'], W['short'], W['max'], W['rand']])   # same draws as the data file
h, s = z[cause == 0], z[cause == 1]
print('share of each cause:', np.bincount(cause) / 1000)
print('sigma of the hit readings:', np.sqrt(((h - 3) ** 2).mean()).round(4))
print('lambda of the short readings (n / sum):', (len(s) / s.sum()).round(3))

share of each cause: [0.733 0.141 0.047 0.079]
sigma of the hit readings: 0.0505
lambda of the short readings (n / sum): 2.078


## 5. Try it: how the weights change the shape

The slider sets the weight of the unexpected-object part; the other three keep their ratios.

In [10]:
zz = np.linspace(0, 4.99, 1500)
fig = go.Figure()
values = [0.0, 0.06, 0.12, 0.24, 0.4]
for i, ws in enumerate(values):
    scale = (1 - ws) / (1 - W['short'])
    w = dict(hit=W['hit'] * scale, short=ws, max=W['max'] * scale, rand=W['rand'] * scale)
    fig.add_trace(go.Scatter(x=zz, y=mixture(zz, 3.0, w), visible=(i == 2), name=f'short weight {ws}'))
steps = [dict(method='update', args=[{'visible': [j == i for j in range(len(values))]}], label=str(v)) for i, v in enumerate(values)]
fig.update_layout(sliders=[dict(steps=steps, active=2, currentvalue=dict(prefix='weight of p_short: '))],
                  yaxis=dict(range=[0, 1.2], title='density (per m)'), xaxis=dict(title='reading z (m)'), template='simple_white')
fig